# НИР3. Анализ кампании r2

Данные — кампания `r2` (коммит стенда 710cbf7, протокол от 2026-09-21):
Apple M4 (Chromium 153, WebKit 26.6) и AMD Ryzen 5 3550H (Chromium 153,
Firefox 155). Правила анализа предрегистрированы в `nir3-plan.md`,
раздел 5; итоги проверки данных — `r2-validation.md`. Разбор первой
основной серии — `archive/main` (в выводы не идёт).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from nirlib import config as C
from nirlib import compare as K
from nirlib import plots as P
from nirlib import quality as Q
from nirlib.load import controls, load

P.setup()
pd.set_option("display.max_columns", 40, "display.width", 200)
for d in (C.FIG_DIR, C.TABLE_DIR, C.DATA_DIR):
    d.mkdir(exist_ok=True)

## 0. Качество данных

Допуск серий (раздел 2 протокола), покрытие матрицы пула, предупреждения
harness и дрейф. Исключение прогонов по предупреждениям задаётся
`C.WARNING_POLICY`.

In [2]:
ser = Q.series()
ser.to_csv(C.TABLE_DIR / "00_series.csv", index=False)
display(Markdown(f"Серий: **{len(ser)}**, допущено: **{int(ser['admitted'].sum())}**, "
                 f"коммитов стенда: **{ser['commit'].nunique()}** ({', '.join(ser['commit'].unique())})."))
ser.drop(columns=["series", "violations", "usable", "dirty", "isolated", "parity", "frame_exact", "complete"])

Серий: **24**, допущено: **24**, коммитов стенда: **1** (710cbf7).

,campaign,platform,condition,scenario,commit,browser,pinned,vsync_uncapped,cells,n_min,n_expected,fatal,retries,drift_spread,warned_runs,excluded_runs,admitted
0,r2,amd-ryzen-5 · chromium,base,s1,710cbf7,153.0.8010.12,True,True,8,10,10,0,0,0.036029,0,0,True
1,r2,amd-ryzen-5 · chromium,base,s2,710cbf7,153.0.8010.12,True,True,5,30,30,0,0,0.033946,0,0,True
2,r2,amd-ryzen-5 · chromium,trace,s2,710cbf7,153.0.8010.12,True,True,5,30,30,0,0,0.027338,0,0,True
3,r2,amd-ryzen-5 · chromium,base,s3,710cbf7,153.0.8010.12,True,True,2,30,30,0,0,NaN,0,0,True
4,r2,amd-ryzen-5 · chromium,base,s4,710cbf7,153.0.8010.12,True,True,4,20,20,0,0,0.022111,0,0,True
5,r2,amd-ryzen-5 · chromium,base,s5,710cbf7,153.0.8010.12,True,True,4,8,8,0,0,NaN,22,0,True
6,r2,amd-ryzen-5 · firefox,base,s1,710cbf7,155.0,True,True,8,10,10,0,0,0.005741,0,0,True
7,r2,amd-ryzen-5 · firefox,base,s2,710cbf7,155.0,True,True,5,30,30,0,0,0.031075,0,0,True
8,r2,amd-ryzen-5 · firefox,base,s3,710cbf7,155.0,True,True,2,30,30,0,0,NaN,0,0,True
9,r2,amd-ryzen-5 · firefox,base,s4,710cbf7,155.0,True,True,4,20,20,0,0,0.064179,0,0,True


In [3]:
cov = Q.coverage(ser)
cov.to_csv(C.TABLE_DIR / "00_coverage.csv", index=False)
cov

,platform,condition,s1,s2,s3,s4,s5
0,apple-m4 · chromium,base,●,●,●,●,●
1,apple-m4 · chromium,trace,,●,,,
2,apple-m4 · webkit,base,,●,●,●,
3,apple-m4 · chromium,cpu×4,,●,●,●,
4,apple-m4 · chromium,slow4g,,,●,,
5,amd-ryzen-5 · chromium,base,●,●,●,●,●
6,amd-ryzen-5 · chromium,trace,,●,,,
7,amd-ryzen-5 · firefox,base,●,●,●,●,●


In [4]:
CONDITIONS = tuple(ser["condition"].unique())
runs_all, _, _ = load(conditions=CONDITIONS, keep_excluded=True)
warn = Q.warnings(runs_all)
warn.to_csv(C.TABLE_DIR / "00_warnings.csv", index=False)
display(Markdown(f"Политика исключения: `{C.WARNING_POLICY}`. "
                 f"Исключено прогонов: **{int((runs_all['exclude_reason'] != '').sum())}** из {len(runs_all)}."))
warn

Политика исключения: `{'vsync не снят': False, 'повтор уровня': False}`. Исключено прогонов: **0** из 2146.

,platform,condition,scenario,label,kind,warned,excluded,runs
0,amd-ryzen-5 · chromium,base,s5,r3f-ref,SN: повтор уровня N отличается от свипа ×N,5,0,8
1,amd-ryzen-5 · chromium,base,s5,r3f-ref-central,SN: повтор уровня N отличается от свипа ×N,6,0,8
2,amd-ryzen-5 · chromium,base,s5,r3f-state,SN: повтор уровня N отличается от свипа ×N,4,0,8
3,amd-ryzen-5 · chromium,base,s5,threejs,SN: повтор уровня N отличается от свипа ×N,7,0,8
4,amd-ryzen-5 · firefox,base,s5,r3f-ref-central,SN: повтор уровня N отличается от свипа ×N,3,0,8
5,amd-ryzen-5 · firefox,base,s5,r3f-state,SN: повтор уровня N отличается от свипа ×N,8,0,8
6,apple-m4 · chromium,base,s5,r3f-ref,SN: повтор уровня N отличается от свипа ×N,7,0,8
7,apple-m4 · chromium,base,s5,r3f-ref-central,SN: повтор уровня N отличается от свипа ×N,7,0,8
8,apple-m4 · chromium,base,s5,r3f-state,SN: повтор уровня N отличается от свипа ×N,2,0,8
9,apple-m4 · chromium,base,s5,threejs,SN: повтор уровня N отличается от свипа ×N,8,0,8


In [5]:
ctl = controls(conditions=CONDITIONS)
drift = Q.order_drift(runs_all)
drift.to_csv(C.TABLE_DIR / "00_order_drift.csv", index=False)
drift[drift["flagged"]]

,platform,condition,scenario,label,load,metric,n,rho,p,flagged
4,amd-ryzen-5 · chromium,base,s1,threejs,A,frame_ms_median,10,-0.721212,1.857316e-02,True
5,amd-ryzen-5 · chromium,base,s1,threejs,B,frame_ms_median,10,0.660606,3.758838e-02,True
23,amd-ryzen-5 · chromium,trace,s2,r3f-ref,-,frame_ms_median,30,0.372497,4.264511e-02,True
40,amd-ryzen-5 · firefox,base,s2,threejs,-,frame_ms_median,30,-0.442368,1.437399e-02,True
53,apple-m4 · chromium,base,s1,r3f-ref,C,frame_ms_median,10,-0.652451,4.087433e-02,True
74,apple-m4 · chromium,cpu×4,s2,r3f-ref,-,frame_ms_median,30,0.633370,1.720514e-04,True
75,apple-m4 · chromium,cpu×4,s2,r3f-ref+parent-state,-,frame_ms_median,30,0.832703,1.147947e-08,True
76,apple-m4 · chromium,cpu×4,s2,r3f-ref-central,-,frame_ms_median,30,0.707466,1.232255e-05,True
77,apple-m4 · chromium,cpu×4,s2,r3f-state,-,frame_ms_median,30,0.651613,9.598284e-05,True
78,apple-m4 · chromium,cpu×4,s2,threejs,-,frame_ms_median,30,0.491490,5.810901e-03,True


## 1. Размер кода и холодный старт (S3)

## 2. Нагрузка на графический процессор (S1)

## 3. Нагрузка на центральный процессор (S2)

## 4. Отзывчивость на ввод (S4)

## 5. Масштабирование (S5)

## 6. Условия: слабый CPU, медленная сеть, WebKit

## 7. Проверка гипотез H1–H3

## 8. Сводка

## Выгрузка датасета

Полный: исключённые прогоны остаются с причиной в `exclude_reason`.

In [6]:
runs, levels, bundles = load(conditions=CONDITIONS, keep_excluded=True)
runs.drop(columns="file").to_csv(C.DATA_DIR / "runs.csv", index=False)
levels.to_csv(C.DATA_DIR / "levels.csv", index=False)
bundles.to_csv(C.DATA_DIR / "bundles.csv", index=False)
runs.groupby(["platform", "condition", "scenario"]).size().unstack(fill_value=0)

scenario                          s1   s2  s3  s4  s5
platform               condition                     
amd-ryzen-5 · chromium base       80  150  60  80  32
                       trace       0  150   0   0   0
amd-ryzen-5 · firefox  base       80  150  60  80  32
apple-m4 · chromium    base       80  150  60  80  32
                       cpu×4       0  150  60  80   0
                       slow4g      0    0  60   0   0
                       trace       0  150   0   0   0
apple-m4 · webkit      base        0  150  60  80   0